# Mutation von Hand

Ein kleiner Parser für Server-Sent Events, nachgebaut nach dem Parser, über den in AstroPlanner die gestreamten Antworten von Marvin kommen. Dazu 61 Tests, die ein Agent aus der Spezifikation geschrieben hat. Alle sind grün. Die Frage ist dieselbe wie auf der Folie: Würde ein Test rot, wenn die Logik fehlt?

In drei Schritten:

1. Spezifikation und Tests ansehen
2. Tests laufen lassen
3. Fehler von Hand einbauen und schauen, ob ein Test rot wird

### Aufgaben

**Für alle** (rund 10 Minuten)

1. Spezifikation und Tests ansehen, Tests laufen lassen *(Einstieg)*
2. Aufwärmen: Fehler 3 einbauen und wieder zurücknehmen
3. Fehler 6 bis 8: vermuten, einbauen, testen. Was haben sie gemeinsam?

**Bonus**

4. Den Test schreiben, der Fehler 6 fängt
5. Knobelaufgabe: Tests, die auch 7 und 8 fangen

Zeitbox 15 Minuten, zu zweit. Jede Aufgabe steht noch einmal an der Stelle, an der ihr sie macht. Jede Zelle läuft in wenigen Sekunden.

## 1. Spezifikation und Tests ansehen

> **Aufgabe 1** (Einstieg): Die Spezifikation und die zwei Tests darunter lesen, dann in Abschnitt 2 die beiden Zellen ausführen (Shift+Enter). Welche Regel der Spezifikation prüft der erste der zwei Tests? Fertig, wenn „Ergebnis: grün (61 Tests)“ dasteht.

Der Browser bekommt die Antwort in Textstücken beliebiger Länge, so wie sie aus dem Netz kommen. Der Parser setzt daraus die Events zusammen. Das ist die Spezifikation, aus der ein Agent die Tests geschrieben hat, test-first, also ohne den Code zu kennen:

In [ ]:
# Nur in Colab: Hilfsdateien und Daten aus dem Repo holen. Lokal tut die Zelle nichts.
import os
import subprocess
import sys

if "google.colab" in sys.modules:
    repo = "/content/ki-workshop-handson"
    if not os.path.exists(repo):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/dprinz/ki-workshop-handson.git", repo], check=True)
    os.chdir(f"{repo}/10-mutation-von-hand")

In [1]:
from IPython.display import Markdown

import helfer

Markdown(open("spezifikation.md", encoding="utf-8").read())

# Spezifikation: Parser für Server-Sent Events

Die Antwort des Assistenten kommt als Strom von Server-Sent Events (SSE). Der Browser liefert
den Strom in Textstücken beliebiger Länge. Der Parser setzt daraus die Events zusammen und
gibt den Inhalt ihres Felds `data` zurück.

## Schnittstelle

Modul `sse_parser`, Klasse `SseParser`, eine Instanz pro Strom.

- `einlesen(stueck: str) -> list[str]`: nimmt das nächste Textstück und gibt die Daten aller
  Events zurück, die mit diesem Stück fertig geworden sind, in der Reihenfolge des Stroms.
  Wurde kein Event fertig, ist die Liste leer.
- `abschliessen() -> list[str]`: am Ende des Stroms. Gibt die Daten eines letzten Events
  zurück, das keine abschliessende Leerzeile mehr bekommen hat, sonst eine leere Liste.
  Leert den Puffer.

## Regeln

1. Ein Event endet mit einer Leerzeile, also mit zwei Zeilenumbrüchen hintereinander.
2. Ein Event kann über mehrere Stücke verteilt ankommen. Auch die Leerzeile am Ende kann auf
   zwei Stücke verteilt sein.
3. Ein Stück kann mehrere Events enthalten.
4. Zusätzliche Leerzeilen zwischen Events werden ignoriert.
5. Zeilenenden `\r\n` gelten wie `\n`.
6. Verwendet wird nur das Feld `data`: Zeilen, die mit `data:` beginnen. Ein Leerzeichen
   direkt nach dem Doppelpunkt gehört nicht zu den Daten. Hat ein Event mehrere `data:`-Zeilen,
   werden ihre Werte mit `\n` verbunden.
7. Andere Felder und Kommentarzeilen (sie beginnen mit `:`) werden ignoriert. Ein Event ohne
   `data:`-Zeile liefert nichts.


Die Tests stehen in `test_sse_parser.py`, geschrieben von Claude Opus 5.5. `unittest` ist das JUnit von Python: Testklassen, Methoden mit `test_` am Anfang, `self.assertEqual(...)` wie `assertEquals` in JUnit. Zwei der 61 Tests:

In [2]:
import inspect

import test_sse_parser

print(inspect.getsource(test_sse_parser.TestVerteilteStuecke.test_event_ueber_mehrere_stuecke_wird_zusammengesetzt))
print(inspect.getsource(test_sse_parser.TestDatenfeld.test_mehrere_datenzeilen_werden_mit_zeilenumbruch_verbunden))

    def test_event_ueber_mehrere_stuecke_wird_zusammengesetzt(self):
        self.assertEqual(self.parser.einlesen("da"), [])
        self.assertEqual(self.parser.einlesen("ta: Hal"), [])
        self.assertEqual(self.parser.einlesen("lo\n"), [])
        self.assertEqual(self.parser.einlesen("\n"), ["Hallo"])

    def test_mehrere_datenzeilen_werden_mit_zeilenumbruch_verbunden(self):
        self.assertEqual(
            self.parser.einlesen("data: eins\ndata: zwei\ndata: drei\n\n"),
            ["eins\nzwei\ndrei"],
        )



Die ganze Datei öffnet ihr links im Dateibrowser mit einem Doppelklick.

## 2. Tests laufen lassen

Die nächste Zelle schreibt den Parser in die Datei `sse_parser.py`. In dieser Zelle baut ihr nachher die Fehler ein. Für Java-Augen: Blöcke entstehen durch Einrückung, `self` ist `this`, und `events.pop()` nimmt das letzte Element aus der Liste und gibt es zurück.

In [3]:
%%writefile sse_parser.py
MAX_BYTES_PRO_EVENT = 64 * 1024


class EventZuGross(Exception):
    pass


class SseParser:
    def __init__(self):
        self.puffer = ""  # Text, der noch nicht mit einer Leerzeile endet

    def einlesen(self, stueck):
        self.puffer = (self.puffer + stueck).replace("\r\n", "\n")
        events = self.puffer.split("\n\n")
        self.puffer = events.pop()  # das letzte Stück ist noch nicht fertig

        for event in events + [self.puffer]:
            if len(event.encode("utf-8")) > MAX_BYTES_PRO_EVENT:
                raise EventZuGross()

        ergebnis = []
        for event in events:
            daten = daten_aus_event(event)
            if daten is not None:
                ergebnis.append(daten)
        return ergebnis

    def abschliessen(self):
        rest = self.puffer
        self.puffer = ""
        daten = daten_aus_event(rest)
        if daten is None:
            return []
        return [daten]


def daten_aus_event(event):
    werte = []
    for zeile in event.split("\n"):
        if zeile.startswith("data:"):
            wert = zeile[len("data:"):]
            if wert.startswith(" "):
                wert = wert[1:]
            werte.append(wert)
    if not werte:
        return None
    return "\n".join(werte)

Overwriting sse_parser.py


In [4]:
helfer.tests_laufen_lassen()

Ergebnis: grün (61 Tests)
Parser: alle acht Stellen der Tabelle wie im Original.


Alle grün, und die Tests taugen etwas: Im ersten Lauf waren drei davon rot, weil der Parser einen echten Fehler hatte. Kam ein `\r\n` genau zwischen zwei Stücken getrennt an, blieb das `\r` in den Daten. Behoben ist das im Parser, die Tests sind unverändert.

Wer alle 61 Testnamen sehen will: `helfer.tests_laufen_lassen(ausfuehrlich=True)`.

## 3. Fehler von Hand einbauen

Ein Versuch geht so:

1. Vermuten: Wird ein Test rot?
2. In der Parser-Zelle oben die Zeile ändern, die Zelle ausführen (Shift+Enter), dann die Testzelle darunter.
3. Rot heisst: Die Tests fangen den Fehler. Grün heisst: Der Fehler käme durch.
4. Die Änderung zurücknehmen und beide Zellen wieder ausführen. Die Testzelle meldet unter dem Ergebnis, ob der Parser an einer Stelle der Tabelle noch verändert ist.

| Nr | Zeile im Parser | ändern zu | Wirkung |
|---|---|---|---|
| 1 | `self.puffer = (self.puffer + stueck).replace("\r\n", "\n")` | `self.puffer = self.puffer + stueck` | `\r\n` wird nicht mehr zu `\n` |
| 2 | `self.puffer = events.pop()` | `events.pop()` | fertige Events bleiben im Puffer und kommen noch einmal |
| 3 | `if wert.startswith(" "):` und die Zeile darunter | beide löschen | das Leerzeichen nach `data:` bleibt in den Daten |
| 4 | `return "\n".join(werte)` | `return "".join(werte)` | mehrere `data:`-Zeilen kleben aneinander |
| 5 | `self.puffer = ""` in `abschliessen` | löschen | der letzte Rest bleibt nach dem Abschliessen im Puffer |
| 6 | `raise EventZuGross()` | `pass` | ein Event über 64 KiB geht durch |
| 7 | `for event in events + [self.puffer]:` | `for event in events:` | ein Event, das nie fertig wird, wächst ohne Grenze |
| 8 | `len(event.encode("utf-8"))` | `len(event)` | die Grenze zählt Zeichen statt Bytes |

Wer eine Zeile ganz löscht und damit einen Block leer macht, schreibt `pass` hin: Python braucht in jedem Block mindestens eine Zeile.

> **Aufgabe 2**: Zum Aufwärmen Fehler 3 einbauen und wieder zurücknehmen. Vermutet zuerst: Wird ein Test rot? Fertig, wenn die Testzelle danach wieder „grün (61 Tests)“ zeigt und darunter „alle acht Stellen der Tabelle wie im Original“.

Fehler 3 heisst: in der Parser-Zelle diese zwei Zeilen löschen, die Zelle ausführen und dann die Testzelle.

```python
            if wert.startswith(" "):
                wert = wert[1:]
```

Zurücknehmen: in der Parser-Zelle Cmd+Z (Mac) oder Ctrl+Z drücken, bis die zwei Zeilen wieder dastehen. Klappt das nicht, die zwei Zeilen von hier kopieren und direkt unter `wert = zeile[len("data:"):]` einfügen, gleich weit eingerückt. Dann Parser-Zelle und Testzelle wieder ausführen.

> **Aufgabe 3**: Jetzt die Fehler 6 bis 8, einer nach dem anderen, jedes Mal mit demselben Ablauf. Vermutet vor dem Ausführen: rot oder grün? Zum Schluss: Was haben die Fehler gemeinsam, die kein Test fängt? Schaut dazu noch einmal in die Spezifikation. Wer Zeit hat, probiert auch 1, 2, 4 und 5 oder eine eigene Idee.

Die Lösung darunter probiert alle acht Fehler auf einmal durch.

In [5]:
# @title Lösung (aufklappen)
# Die Zelle probiert alle acht Fehler automatisch durch, jeweils in einer Kopie des Parsers.
# Im Kleinen ist das Mutation Testing, wie es Werkzeuge systematisch machen, für Java etwa PIT.
#
# Gefangen werden 1 bis 5. Das steht alles in der Spezifikation, und dafür hat der Agent Tests geschrieben.
# Nicht gefangen werden 6 bis 8. Alle drei betreffen die Obergrenze von 64 KiB pro Event.
# Die steht im Code, aber nicht in der Spezifikation. Also prüft sie kein Test,
# und man könnte sie ganz löschen, ohne dass etwas rot wird.
# Fehler 3 macht 48 der 61 Tests rot: Die meisten Tests schicken "data: …" mit Leerzeichen.
helfer.mutationen_pruefen()

Tests: test_sse_parser.py

1  \r\n nicht mehr umwandeln                      gefangen         rot (10 von 61 Tests)
2  Rest nicht im Puffer behalten                  gefangen         rot (9 von 61 Tests)


3  Leerzeichen nach data: nicht entfernen         gefangen         rot (48 von 61 Tests)
4  Datenzeilen ohne Zeilenumbruch verbinden       gefangen         rot (11 von 61 Tests)


5  Puffer beim Abschliessen nicht leeren          gefangen         rot (2 von 61 Tests)


6  Grenze nicht mehr durchsetzen                  nicht gefangen   grün (61 Tests)
7  nur fertige Events prüfen, den Puffer nicht    nicht gefangen   grün (61 Tests)
8  Zeichen statt Bytes zählen                     nicht gefangen   grün (61 Tests)


## Bonus: den fehlenden Test schreiben

> **Aufgabe 4** (Bonus): In der nächsten Zelle einen Test ergänzen, der Fehler 6 fängt: Ein Event über 64 KiB löst `EventZuGross` aus. Zelle ausführen, dann die Zelle darunter. Sie lässt euren Test zusammen mit den 61 generierten gegen alle acht Fehler laufen. Fertig, wenn bei Fehler 6 „gefangen“ steht. Und: Warum hat der Agent diesen Test nicht geschrieben, obwohl er 61 andere geschrieben hat?

<details><summary>Tipp 1</summary>

Ein Test, der eine Exception erwartet, beginnt in `unittest` mit `with self.assertRaises(EventZuGross):`, darunter eingerückt der Aufruf, wie `assertThrows` in JUnit. 64 KiB sind 65'536 Bytes; `"x" * 70_000` ist ein Text aus 70'000 x.

</details>
<details><summary>Tipp 2</summary>

Unter den Kommentar in der Zelle, gleich weit eingerückt wie der Test darüber:

```python
    def test_grosses_event(self):
        with self.assertRaises(EventZuGross):
            SseParser().einlesen("data: " + … + "\n\n")
```

</details>

In [6]:
%%writefile test_grenze.py
import unittest

from sse_parser import EventZuGross, SseParser


class TestGrenze(unittest.TestCase):
    def test_kleines_event_geht_durch(self):
        parser = SseParser()
        # "x" * 1000 ist ein Text aus 1000 x
        self.assertEqual(parser.einlesen("data: " + "x" * 1000 + "\n\n"), ["x" * 1000])

    # Aufgabe 4: ein Test, bei dem ein Event über 64 KiB EventZuGross auslöst
    # Aufgabe 5: Tests, die auch Fehler 7 und 8 fangen

Overwriting test_grenze.py


In [7]:
helfer.tests_laufen_lassen("test_grenze.py")
print()
helfer.mutationen_pruefen("test_sse_parser.py", "test_grenze.py")

Ergebnis: grün (1 Test)
Parser: alle acht Stellen der Tabelle wie im Original.

Tests: test_sse_parser.py, test_grenze.py



1  \r\n nicht mehr umwandeln                      gefangen         rot (10 von 62 Tests)


2  Rest nicht im Puffer behalten                  gefangen         rot (9 von 62 Tests)
3  Leerzeichen nach data: nicht entfernen         gefangen         rot (49 von 62 Tests)


4  Datenzeilen ohne Zeilenumbruch verbinden       gefangen         rot (11 von 62 Tests)


5  Puffer beim Abschliessen nicht leeren          gefangen         rot (2 von 62 Tests)
6  Grenze nicht mehr durchsetzen                  nicht gefangen   grün (62 Tests)


7  nur fertige Events prüfen, den Puffer nicht    nicht gefangen   grün (62 Tests)


8  Zeichen statt Bytes zählen                     nicht gefangen   grün (62 Tests)


> **Aufgabe 5** (Bonus, Knobelaufgabe): Fängt euer Test auch 7 und 8? Wenn nicht: in derselben Zelle Tests ergänzen, bis bei 6, 7 und 8 „gefangen“ steht. Was die beiden Fehler ändern, steht in der Spalte „Wirkung“ der Tabelle. Noch knapper: Es geht mit einem einzigen Test.

<details><summary>Tipp 1</summary>

Fehler 7 prüft nur Events, die mit einer Leerzeile fertig sind. Gebraucht wird also ein Event, das über 64 KiB wächst und nie fertig wird. Fehler 8 zählt Zeichen statt Bytes. Gebraucht wird also ein Text mit weniger als 65'536 Zeichen, aber mehr als 65'536 Bytes.

</details>
<details><summary>Tipp 2</summary>

`ä` braucht in UTF-8 zwei Bytes: `"ä" * 40_000` hat 40'000 Zeichen und 80'000 Bytes. Und ein Stück ohne `"\n\n"` am Ende macht kein Event fertig, etwa `parser.einlesen("data: " + …)`.

</details>

In [8]:
# @title Lösung (aufklappen)
# Warum hat der Agent den Test nicht geschrieben? Er hat aus der Spezifikation gearbeitet, und dort steht keine
# Obergrenze. Ein Test prüft nur, was in seinen Assertions steht. Stammen die aus der Spezifikation,
# hat er dieselben Lücken.
#
# In AstroPlanner war es genauso: Alle generierten Parser-Tests waren grün, die Spezifikation kannte keine
# Obergrenze, und im Code fehlte sie auch. Den unbegrenzten Puffer hat erst der Security-Reviewer gefunden.
# Danach kam die Regel "Antwort zu lang" in die Spezifikation (20'000 Zeichen pro Antwort, 64 KiB pro Event),
# und erst dann entstanden die Tests dafür.
#
# Ein Test mit einem fertigen Event aus 70'000 x fängt 6, aber nicht 7 und 8:
# 7 braucht ein Event, das nie fertig wird, 8 Zeichen, die in UTF-8 mehr als ein Byte brauchen.
# Mit einem Test geht es auch: SseParser().einlesen("data: " + "ä" * 40_000), ohne Leerzeile am Ende.
# Das Event wird nie fertig (7), hat 40'006 Zeichen, aber 80'006 Bytes (8), und ohne Grenze geht es durch (6).
from pathlib import Path

Path("test_grenze_loesung.py").write_text(r'''
import unittest

from sse_parser import EventZuGross, SseParser


class TestGrenzeLoesung(unittest.TestCase):
    def test_fertiges_event_ueber_64_kib(self):
        with self.assertRaises(EventZuGross):
            SseParser().einlesen("data: " + "x" * 70_000 + "\n\n")

    def test_event_das_nie_fertig_wird(self):
        parser = SseParser()
        parser.einlesen("data: ")
        with self.assertRaises(EventZuGross):
            for _ in range(10):
                parser.einlesen("x" * 10_000)

    def test_grenze_zaehlt_bytes(self):
        # 40'000 Zeichen, aber 80'000 Bytes: ä braucht in UTF-8 zwei Bytes
        with self.assertRaises(EventZuGross):
            SseParser().einlesen("data: " + "ä" * 40_000 + "\n\n")
''', encoding="utf-8")

helfer.tests_laufen_lassen("test_grenze_loesung.py")
print()
helfer.mutationen_pruefen("test_sse_parser.py", "test_grenze_loesung.py")

Ergebnis: grün (3 Tests)
Parser: alle acht Stellen der Tabelle wie im Original.



Tests: test_sse_parser.py, test_grenze_loesung.py



1  \r\n nicht mehr umwandeln                      gefangen         rot (10 von 64 Tests)
2  Rest nicht im Puffer behalten                  gefangen         rot (9 von 64 Tests)


3  Leerzeichen nach data: nicht entfernen         gefangen         rot (48 von 64 Tests)


4  Datenzeilen ohne Zeilenumbruch verbinden       gefangen         rot (11 von 64 Tests)


5  Puffer beim Abschliessen nicht leeren          gefangen         rot (2 von 64 Tests)


6  Grenze nicht mehr durchsetzen                  gefangen         rot (3 von 64 Tests)


7  nur fertige Events prüfen, den Puffer nicht    gefangen         rot (1 von 64 Tests)
8  Zeichen statt Bytes zählen                     gefangen         rot (1 von 64 Tests)


## Was ihr gesehen habt

Die 61 generierten Tests fangen jeden Fehler, der ein Verhalten aus der Spezifikation bricht, und keinen an der Obergrenze, die nur im Code steht. Ein Test prüft nur, was in seinen Assertions steht; stammen die aus der Spezifikation, hat er dieselben Lücken. Werkzeuge für Mutation Testing, für Java etwa PIT, bauen solche Fehler automatisch ein und melden, welche kein Test fängt.